# Example 8.1.2 / 1: Convexity Changes the Meaning

Companion notebook for **8.1.2 Nonlinear and Global Solvers** in the AMPL lab Documentation, section "Convexity Changes the Meaning". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Convexity Changes the Meaning

For a convex minimization problem with appropriate regularity, a local optimum is global. For
a nonconvex model, different starting points can lead a local solver to different solutions.
Finite bounds are especially important for many global reformulations and relaxations.

The function $f(x) = x^4 - 3x^2 + x$ on $-2 \le x \le 2$ is nonconvex, with one local minimum
near $x = 1.13$ and a lower, global one near $x = -1.30$. The example starts Ipopt from
$x = 1$ and from $x = -1$, then runs Couenne. In `run.py`, `ampl` is an `AMPL` object that has
already read `model.mod`; `ampl.var["x"].set_value(...)` sets the starting point and
`verbose=False` hides the solver logs:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;                     # one decision, with finite bounds
minimize f: x^4 - 3 * x^2 + x;         # nonconvex: two local minima

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for solver, start in [("ipopt", 1), ("ipopt", -1), ("couenne", 1)]:
    ampl.var["x"].set_value(start)
    ampl.solve(solver=solver, verbose=False)
    x = ampl.var["x"].value()
    print(f"{solver} from {start}: {ampl.solve_result}, x = {x:.4f}, f = {ampl.obj['f'].value():.4f}")

ipopt from 1: solved, x = 1.1309, f = -1.0702
ipopt from -1: solved, x = -1.3008, f = -3.5139
couenne from 1: solved, x = -1.3008, f = -3.5139


### Pure AMPL: model.run

The same Ipopt runs in pure AMPL, with the same `model.mod`. `let` sets the starting point.
Couenne prints a long log that its AMPL options cannot silence, so it is left out here; it runs
with `option solver couenne; solve;`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver_msg 0;
option ipopt_options 'print_level=0';
option solver ipopt;
let x := 1;   solve;  display x, f;
let x := -1;  solve;  display x, f;

Ipopt 3.12.13: print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************


suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
x = 1.1309
f = -1.07023

Ipopt 3.12.13: print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

x = -1.30084
f = -3.51391

